In [1]:
"""
NLP Project: Dimensional Aspect Sentiment Regression (DimASR)
Final Code
"""

# Using transformers for BERT model, datasets for handling, and scikit-learn for baseline
!pip install -q transformers datasets scikit-learn accelerate

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import os

# Setting a universal seed for reproducibility of results
torch.manual_seed(42)
np.random.seed(42)

# Checking for GPU availability and setting the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# DATASET ACQUISITION, PARSING, AND PREPARATION
# URLs for the official SemEval 2026 DimASR dataset
TRAIN_URL = "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_1/eng/eng_restaurant_train_alltasks.jsonl"
DEV_URL = "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_1/eng/eng_restaurant_dev_task1.jsonl"

# Downloading the data files directly into the Colab environment
!wget -q {TRAIN_URL} -O train.jsonl
!wget -q {DEV_URL} -O dev.jsonl

def parse_va_string(va_str: str) -> tuple[float, float]:
    """Helper to parse 'V#A' strings into float tuples."""
    try:
        v, a = va_str.split('#')
        return float(v), float(a)
    except (ValueError, AttributeError):
        # Returning a neutral value if the string is malformed or missing
        return 5.0, 5.0

def load_and_normalize_data(file_path: str, is_train_file: bool = False) -> pd.DataFrame:
    """
    Parses JSONL files and normalizes the different structures (Quadruplet vs. Aspect_VA).
    This function addresses the "Data Format Discrepancy" anticipated problem.
    """
    data_list = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            # Conditional logic to handle the different data structures
            items = obj.get('Quadruplet', []) if is_train_file else obj.get('Aspect_VA', [])

            for item in items:
                # Filtering "NULL" aspects to focus on explicit aspect regression
                if item.get('Aspect') == "NULL":
                    continue

                valence, arousal = parse_va_string(item.get('VA', "5.0#5.0"))

                data_list.append({
                    'text': obj['Text'],
                    'aspect': item['Aspect'],
                    'labels': [valence, arousal]  # The target for regression models
                })
    return pd.DataFrame(data_list)

print("Loading and normalizing datasets:")
# The train file has the "Quadruplet" key, so passing flag
train_df = load_and_normalize_data('train.jsonl', is_train_file=True)
dev_df = load_and_normalize_data('dev.jsonl', is_train_file=False)
print(f"Loaded {len(train_df)} training samples and {len(dev_df)} development samples")
print("\nSample of processed training data:")
print(train_df.head(2))

# BASELINE MODEL = SVR WITH TF-IDF FEATURES
# This model establishes a performance lower bound using classic feature engineering
print("\nTraining Baseline SVR Model:")

# Creating a simple feature representation by concatenating the text and aspect
train_features = train_df['text'] + " [SEP] " + train_df['aspect']
dev_features = dev_df['text'] + " [SEP] " + dev_df['aspect']

# Initializing a TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))

# Vectorizing the text data
X_train = vectorizer.fit_transform(train_features)
X_dev = vectorizer.transform(dev_features)

# Training two separate SVRs: one for Valence and one for Arousal
svr_valence = make_pipeline(StandardScaler(with_mean=False), SVR())
svr_arousal = make_pipeline(StandardScaler(with_mean=False), SVR())

print("Fitting Valence SVR:")
svr_valence.fit(X_train, train_df['labels'].apply(lambda x: x[0]))
print("Fitting Arousal SVR:")
svr_arousal.fit(X_train, train_df['labels'].apply(lambda x: x[1]))

# Making predictions on the development set
pred_v = svr_valence.predict(X_dev)
pred_a = svr_arousal.predict(X_dev)

# Evaluating using the official RMSE metric
baseline_rmse = np.sqrt(mean_squared_error(
    y_true=dev_df['labels'].tolist(),
    y_pred=list(zip(pred_v, pred_a))
))
print(f"Baseline SVR RMSE: {baseline_rmse:.4f}")

# ADVANCED MODEL = FINE-TUNING BERT FOR REGRESSION
# This model leverages Transformers for deep contextual understanding
print("\nPreparing Advanced BERT Model:")

# Choosing "bert-base-cased" to preserve capitalization, which is a signal for Arousal
MODEL_CKPT = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

class DimASRDataset(Dataset):
    """PyTorch Dataset class for handling the tokenization and formatting"""
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        # This input format addresses the "Aspect Cross-Talk" problem
        encoding = self.tokenizer(
            row['text'],
            row['aspect'],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(row['labels'], dtype=torch.float)
        }

# Creating Dataset objects for the trainer
train_dataset = DimASRDataset(train_df, tokenizer)
dev_dataset = DimASRDataset(dev_df, tokenizer)

# Initializing the model with a regression head (num_labels=2)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=2
).to(device)

def compute_metrics(eval_pred):
    """Custom metric function to compute RMSE during evaluation"""
    predictions, labels = eval_pred
    # The output from the model is unbounded; so, must clip it to the valid range
    predictions = np.clip(predictions, 1.0, 9.0)
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    return {"rmse": rmse}

# Training Arguments, including regularization to prevent overfitting
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.1,  # Solution for "Capacity Mismatch"
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True, # Solution for "Capacity Mismatch" via Early Stopping
    metric_for_best_model="rmse",
    greater_is_better=False,
    logging_steps=100,
    fp16=torch.cuda.is_available(), # Using mixed-precision if on GPU
)

class CustomTrainer(Trainer):
    """Custom Trainer to implement Huber Loss for robustness to outliers."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # This addresses the "Regression to the Mean" problem
        loss_fct = nn.SmoothL1Loss() # Huber Loss
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1, self.model.config.num_labels))
        return (loss, outputs) if return_outputs else loss

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

print("BERT fine-tuning:")
trainer.train()

# FINAL EVALUATION AND ANALYSIS
print("\nFinal Model Evaluation:")
metrics = trainer.evaluate()
print(f"Fine-tuned BERT RMSE: {metrics['eval_rmse']:.4f}")

# Displaying a qualitative comparison of predictions
print("\nSample Prediction Analysis:")
predictions = trainer.predict(dev_dataset)
preds = np.clip(predictions.predictions, 1.0, 9.0)

print(f"{'Aspect':<20} | {'True V/A':<12} | {'Predicted V/A':<15}")
print("-" * 55)
for i in range(5):
    aspect = dev_df.iloc[i]['aspect']
    true_v, true_a = dev_df.iloc[i]['labels']
    pred_v, pred_a = preds[i]
    print(f"{aspect:<20} | {true_v:.2f}#{true_a:.2f}      | {pred_v:.2f}#{pred_a:.2f}")


Using device: cuda
Loading and normalizing datasets:
Loaded 2779 training samples and 340 development samples

Sample of processed training data:
                                                text           aspect  \
0  their sake list was extensive , but we were lo...        sake list   
1  the spicy tuna roll was unusually good and the...  spicy tuna roll   

        labels  
0  [7.83, 8.0]  
1  [7.5, 7.62]  

Training Baseline SVR Model:
Fitting Valence SVR:
Fitting Arousal SVR:
Baseline SVR RMSE: 1.1465

Preparing Advanced BERT Model:


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

...

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte


Final Model Evaluation:


Training Loss,Validation Loss,Epoch,Rmse
0.334791,0.311642,3,0.891246


Fine-tuned BERT RMSE: 0.8912

Sample Prediction Analysis:


Aspect               | True V/A     | Predicted V/A  
-------------------------------------------------------
diner food           | 7.25#6.75      | 7.75#7.68
breakfast            | 7.25#6.75      | 7.66#7.64
food                 | 7.50#7.75      | 7.42#7.48
drinks               | 7.50#7.50      | 7.40#7.46
service              | 7.75#7.75      | 7.38#7.46


In [2]:
# INFERENCE ON OFFICIAL TEST SET FOR SUBMISSION
print("\nRunning Inference on Test Set:")

# URL for the official SemEval 2026 DimASR test dataset
TEST_URL = "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_1/eng/eng_restaurant_test_task1.jsonl"

# Downloading the test file directly into the Colab environment
!wget -q {TEST_URL} -O test.jsonl

print("Loading and Normalizing Test Dataset:")
# Using is_train_file=False because the test set uses the "Aspect_VA" format, not "Quadruplet"
test_df = load_and_normalize_data('test.jsonl', is_train_file=False)
print(f"Loaded {len(test_df)} test samples.")

# Creating the Dataset object for the trainer using the class defined earlier
test_dataset = DimASRDataset(test_df, tokenizer)

print("Generating predictions with fine-tuned BERT:")
# Generating predictions using the trainer
test_outputs = trainer.predict(test_dataset)

# The model outputs unbounded regression values; so, must clip them to the official 1.0 - 9.0 range
test_preds = np.clip(test_outputs.predictions, 1.0, 9.0)

# Adding the predictions back to dataframe for easy viewing and exporting
test_df['Predicted_Valence'] = test_preds[:, 0]
test_df['Predicted_Arousal'] = test_preds[:, 1]

# Displaying a qualitative sample of the final predictions for the report
print("\nSample Test Predictions:")
print(f"{'Text Snippet':<30} | {'Aspect':<15} | {'Predicted V#A':<15}")

for i in range(min(5, len(test_df))):
    # Truncating text for cleaner printing
    text_snip = test_df.iloc[i]['text'][:27] + "..." if len(test_df.iloc[i]['text']) > 30 else test_df.iloc[i]['text']
    aspect = test_df.iloc[i]['aspect']
    pred_v = test_df.iloc[i]['Predicted_Valence']
    pred_a = test_df.iloc[i]['Predicted_Arousal']
    print(f"{text_snip:<30} | {aspect:<15} | {pred_v:.2f}#{pred_a:.2f}")

# Saving the results to a CSV file
submission_file = "semeval_2026_test_predictions.csv"
test_df[['text', 'aspect', 'Predicted_Valence', 'Predicted_Arousal']].to_csv(submission_file, index=False)
print(f"\nPredictions successfully saved to {submission_file}")



Running Inference on Test Set:
Loading and Normalizing Test Dataset:
Loaded 1504 test samples.
Generating predictions with fine-tuned BERT:



Sample Test Predictions:
Text Snippet                   | Aspect          | Predicted V#A  
A friend suggested this caf... | cafe            | 4.03#5.76
The beer selection is secon... | beer selection  | 4.07#5.76
It was pretty bland for my ... | pepper pizza    | 4.32#6.05
It was pretty bland for my ... | pepperoni       | 4.43#6.10
It was pretty bland for my ... | sausage         | 4.25#6.04

Predictions successfully saved to semeval_2026_test_predictions.csv
